In [1]:
import pandas as pd

1 - First We load the sheet and skip the first few rows as they are not part of the table, and assign the first row as header and resetting index of the data frame:

In [2]:
df = pd.read_excel("Transaction_2026-02-01_2026-02-08.xlsx")
df = df.tail(len(df)-6).reset_index(drop=True)
df.columns = df.loc[0]
df.drop(index=0,inplace=True)
df.reset_index(drop=True,inplace=True)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   First Name                177 non-null    object
 1   Last Name                 177 non-null    object
 2   ID                        177 non-null    object
 3   Department                177 non-null    object
 4   Attendance Group          177 non-null    object
 5   Date                      177 non-null    object
 6   Week                      177 non-null    object
 7   Time                      177 non-null    object
 8   Skin-Surface Temperature  177 non-null    object
 9   Temperature Status        177 non-null    object
 10  Card Swiping Type         177 non-null    object
 11  Verification Method       177 non-null    object
 12  Attendance Check Point    177 non-null    object
 13  Custom Name               177 non-null    object
 14  Data Source               

2 - Converting the time column to actual date time format

In [4]:
df["time_formatted"] = pd.to_datetime(df["Time"],format='%H:%M')

just a sample of the data so we understand what we are working with

In [5]:
df.head()

,First Name,Last Name,ID,Department,Attendance Group,Date,Week,Time,Skin-Surface Temperature,Temperature Status,Card Swiping Type,Verification Method,Attendance Check Point,Custom Name,Data Source,Correction Type,Note,time_formatted
0,employee1_fn,employee1_ln,8,All Departments>O&M,-,2026-02-05,Thursday,16:17,Unknown Skin-Surface Temperature,Unknown,Undefined,Fingerprint,TRUNSTILE PV 1-Cardreader 02,-,Transactions on Device,-,-,1900-01-01 16:17:00
1,employee1_fn,employee1_ln,8,All Departments>O&M,-,2026-02-05,Thursday,16:15,Unknown Skin-Surface Temperature,Unknown,Undefined,Fingerprint,Amenty Admin-Cardreader 01,-,Transactions on Device,-,-,1900-01-01 16:15:00
2,employee1_fn,employee1_ln,8,All Departments>O&M,-,2026-02-05,Thursday,15:27,Unknown Skin-Surface Temperature,Unknown,Undefined,Face,SS Gate 1-Cardreader 01,-,Transactions on Device,-,-,1900-01-01 15:27:00
3,employee1_fn,employee1_ln,8,All Departments>O&M,-,2026-02-05,Thursday,15:23,Unknown Skin-Surface Temperature,Unknown,Undefined,Face,Control ROOM Amenty-Cardreader 01,-,Transactions on Device,-,-,1900-01-01 15:23:00
4,employee1_fn,employee1_ln,8,All Departments>O&M,-,2026-02-05,Thursday,14:01,Unknown Skin-Surface Temperature,Unknown,Undefined,Fingerprint,Amenty Admin-Cardreader 01,-,Transactions on Device,-,-,1900-01-01 14:01:00


3 - To extract the check in/ check out time we need to create a pivot table using min/max on time values, aggregating by employee and day:

In [6]:
df_pvt = df.pivot_table(values="Time",index=["First Name","Last Name","ID","Department","Date"],aggfunc=["min","max"])
df_pvt.head()

min    max
0                                                             Time   Time
First Name   Last Name    ID Department          Date                    
employee1_fn employee1_ln 8  All Departments>O&M 2026-02-01  08:08  16:10
                                                 2026-02-02  08:22  16:08
                                                 2026-02-03  08:22  16:14
                                                 2026-02-04  08:12  16:25
                                                 2026-02-05  08:33  16:17

4 - Now, we convert the pivot table into a regular dataframe by dropping the extra level of row headers:

In [7]:
df_pvt.columns = ['min_time', 'max_time']
df_pvt.reset_index(inplace=True)
df_pvt.head()

,First Name,Last Name,ID,Department,Date,min_time,max_time
0,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-01,08:08,16:10
1,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-02,08:22,16:08
2,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-03,08:22,16:14
3,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-04,08:12,16:25
4,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-05,08:33,16:17


5 - Now the most important step, is to actually utilize this check in/out data in the actual data frame, to do this we simply create a unique id column by concatenating every thing in both tables

In [8]:
df["conc"] = df["First Name"]+df["Last Name"]+df['ID']+df['Department']+df['Date']+df['Time']
df_pvt["conc_in"] = df_pvt["First Name"]+df_pvt["Last Name"]+df_pvt['ID']+df_pvt['Department']+df_pvt['Date']+df_pvt['min_time']
df_pvt["conc_out"] = df_pvt["First Name"]+df_pvt["Last Name"]+df_pvt['ID']+df_pvt['Department']+df_pvt['Date']+df_pvt['max_time']

6 - and then using .merge() method to get the rest of the data from the original table (like gate names and verification methods)

In [9]:
av_in = df_pvt.merge(df,how="left",left_on="conc_in",right_on="conc").drop_duplicates(subset="conc",keep="first").reset_index(drop=True)[["Verification Method","Attendance Check Point","time_formatted"]].rename({"Verification Method":"Verification Method_in","Attendance Check Point":"Attendance Check Point_in","time_formatted":"time_formatted_in"},axis=1)
av_out = df_pvt.merge(df,how="left",left_on="conc_out",right_on="conc").drop_duplicates(subset="conc",keep="first").reset_index(drop=True)[["Verification Method","Attendance Check Point","time_formatted"]].rename({"Verification Method":"Verification Method_out","Attendance Check Point":"Attendance Check Point_out","time_formatted":"time_formatted_out"},axis=1)

df_final = pd.concat([df_pvt,av_in,av_out],axis=1)

df_final.head()

,First Name,Last Name,ID,Department,Date,min_time,max_time,conc_in,conc_out,Verification Method_in,Attendance Check Point_in,time_formatted_in,Verification Method_out,Attendance Check Point_out,time_formatted_out
0,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-01,08:08,16:10,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:08:00,Fingerprint,TRUNSTILE PV 1-Cardreader 02,1900-01-01 16:10:00
1,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-02,08:22,16:08,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:22:00,Fingerprint,TRUNSTILE PV 1-Cardreader 02,1900-01-01 16:08:00
2,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-03,08:22,16:14,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:22:00,Fingerprint,TurnstilePV2-Cardreader 02,1900-01-01 16:14:00
3,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-04,08:12,16:25,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TurnstilePV2-Cardreader 01,1900-01-01 08:12:00,Fingerprint,TurnstilePV2-Cardreader 02,1900-01-01 16:25:00
4,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-05,08:33,16:17,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:33:00,Fingerprint,TRUNSTILE PV 1-Cardreader 02,1900-01-01 16:17:00


7 - Now it is just a matter of calculating a few parameters and then we are done, so, first we calculate the middle hour of each employee so we can find the shift they belong to

In [10]:
df_final["middle_hour"] = (df_final["time_formatted_in"].dt.hour+df_final["time_formatted_out"].dt.hour)/2

8 - Then define a simple function to actually assign a shift for them, and using .apply() method to create a column for it.

In [11]:
def shift(x):
    if x > 6 or x <= 14:
        return "morning shift"
    elif x > 14 or x <=22:
        return "Noon"
    elif x > 22 or x < 6:
        return "Night"

df_final["shift"] = df_final["middle_hour"].apply(shift)
df_final.head()

,First Name,Last Name,ID,Department,Date,min_time,max_time,conc_in,conc_out,Verification Method_in,Attendance Check Point_in,time_formatted_in,Verification Method_out,Attendance Check Point_out,time_formatted_out,middle_hour,shift
0,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-01,08:08,16:10,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:08:00,Fingerprint,TRUNSTILE PV 1-Cardreader 02,1900-01-01 16:10:00,12.0,morning shift
1,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-02,08:22,16:08,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:22:00,Fingerprint,TRUNSTILE PV 1-Cardreader 02,1900-01-01 16:08:00,12.0,morning shift
2,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-03,08:22,16:14,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:22:00,Fingerprint,TurnstilePV2-Cardreader 02,1900-01-01 16:14:00,12.0,morning shift
3,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-04,08:12,16:25,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TurnstilePV2-Cardreader 01,1900-01-01 08:12:00,Fingerprint,TurnstilePV2-Cardreader 02,1900-01-01 16:25:00,12.0,morning shift
4,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-05,08:33,16:17,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:33:00,Fingerprint,TRUNSTILE PV 1-Cardreader 02,1900-01-01 16:17:00,12.0,morning shift


9 - Calculating exact working duration using dt operators and simple mathematical formula

In [12]:
df_final["total_WHs"] = (df_final["time_formatted_out"].dt.hour-df_final["time_formatted_in"].dt.hour) + (df_final["time_formatted_out"].dt.minute-df_final["time_formatted_in"].dt.minute)/60
df_final["total_WHs"] = df_final["total_WHs"].round(2)
df_final.head()
# you might notice some zero values, thats because this is just a sample of data and some days are not fully present 

,First Name,Last Name,ID,Department,Date,min_time,max_time,conc_in,conc_out,Verification Method_in,Attendance Check Point_in,time_formatted_in,Verification Method_out,Attendance Check Point_out,time_formatted_out,middle_hour,shift,total_WHs
0,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-01,08:08,16:10,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:08:00,Fingerprint,TRUNSTILE PV 1-Cardreader 02,1900-01-01 16:10:00,12.0,morning shift,8.03
1,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-02,08:22,16:08,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:22:00,Fingerprint,TRUNSTILE PV 1-Cardreader 02,1900-01-01 16:08:00,12.0,morning shift,7.77
2,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-03,08:22,16:14,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:22:00,Fingerprint,TurnstilePV2-Cardreader 02,1900-01-01 16:14:00,12.0,morning shift,7.87
3,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-04,08:12,16:25,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TurnstilePV2-Cardreader 01,1900-01-01 08:12:00,Fingerprint,TurnstilePV2-Cardreader 02,1900-01-01 16:25:00,12.0,morning shift,8.22
4,employee1_fn,employee1_ln,8,All Departments>O&M,2026-02-05,08:33,16:17,employee1_fnemployee1_ln8All Departments>O&M20...,employee1_fnemployee1_ln8All Departments>O&M20...,Fingerprint,TRUNSTILE PV 1-Cardreader 01,1900-01-01 08:33:00,Fingerprint,TRUNSTILE PV 1-Cardreader 02,1900-01-01 16:17:00,12.0,morning shift,7.73


10 -  And finally, we drop the "technical" columns like concs and formatted time, and renaming the rest and export the final result for analysis:

In [13]:
df_final.columns

Index(['First Name', 'Last Name', 'ID', 'Department', 'Date', 'min_time',
       'max_time', 'conc_in', 'conc_out', 'Verification Method_in',
       'Attendance Check Point_in', 'time_formatted_in',
       'Verification Method_out', 'Attendance Check Point_out',
       'time_formatted_out', 'middle_hour', 'shift', 'total_WHs'],
      dtype='object')

In [14]:
df_final.columns = ['First Name', 'Last Name', 'ID', 'Department', 'Date', 'check_in_time',
       'check_out_time', 'conc_in', 'conc_out', 'Verification Method_in',
       'Attendance Check Point_in', 'time_formatted_in',
       'Verification Method_out', 'Attendance Check Point_out',
       'time_formatted_out', 'middle_hour', 'shift', 'total_WHs']

df_final = df_final[['First Name', 'Last Name', 'ID', 'Department', 'Date', 'check_in_time', 'check_out_time', 'Verification Method_in', 'Attendance Check Point_in', 'Verification Method_out', 'Attendance Check Point_out', 'shift', 'total_WHs']]


df_final.to_excel("output.xlsx", index=False)